# Leukemia Detection — InceptionV3 (No Augmentation, No Callbacks)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# 2. Dataset paths
train_dir = '/kaggle/input/cnmc-leukemia/C-NMC_Leukemia/fold_0/fold_0'
val_dir   = '/kaggle/input/cnmc-leukemia/C-NMC_Leukemia/fold_1/fold_1'
test_dir  = '/kaggle/input/cnmc-leukemia/C-NMC_Leukemia/fold_2/fold_2'

for name, path in [('Train', train_dir), ('Validation', val_dir), ('Test', test_dir)]:
    if os.path.exists(path):
        print(f'\n{name} set:')
        for cls in sorted(os.listdir(path)):
            cls_path = os.path.join(path, cls)
            if os.path.isdir(cls_path):
                print(f'  {cls}: {len(os.listdir(cls_path))} images')

In [ ]:
# 3. Data generators — rescaling ONLY (299x299)
IMG_SIZE = (299, 299)
BATCH_SIZE = 32
datagen = ImageDataGenerator(rescale=1.0/255)
train_gen = datagen.flow_from_directory(train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=True)
val_gen = datagen.flow_from_directory(val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)
test_gen = datagen.flow_from_directory(test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)
print(f'Class mapping: {train_gen.class_indices}')

In [ ]:
# 4. Build model
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
base_model.trainable = False
model = Sequential([base_model, GlobalAveragePooling2D(), BatchNormalization(), Dense(256, activation='relu'), Dropout(0.5), Dense(1, activation='sigmoid')])
model.summary()

In [ ]:
# 5. Phase 1 — NO callbacks, fixed 10 epochs
model.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])
print('PHASE 1: Feature Extraction (No Augmentation, No Callbacks)'); print('='*55)
history_phase1 = model.fit(train_gen, epochs=10, validation_data=val_gen)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history_phase1.history['accuracy'], label='Train'); ax1.plot(history_phase1.history['val_accuracy'], label='Val')
ax1.set_title('Phase 1 — Accuracy (No Aug, No CB)', fontweight='bold'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(history_phase1.history['loss'], label='Train'); ax2.plot(history_phase1.history['val_loss'], label='Val')
ax2.set_title('Phase 1 — Loss (No Aug, No CB)', fontweight='bold'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Best Val Accuracy: {max(history_phase1.history["val_accuracy"])*100:.2f}%')
print(f'Last Val Accuracy: {history_phase1.history["val_accuracy"][-1]*100:.2f}%')

In [ ]:
# 7. Unfreeze from mixed7
base_model.trainable = True
for i, layer in enumerate(base_model.layers):
    if layer.name == 'mixed7': fine_tune_from = i; break
for layer in base_model.layers[:fine_tune_from]: layer.trainable = False
print(f'Trainable: {sum(1 for l in base_model.layers if l.trainable)} | Frozen: {sum(1 for l in base_model.layers if not l.trainable)}')

In [ ]:
# 8. Phase 2 — NO callbacks, fixed 10 epochs
model.compile(optimizer=Adam(learning_rate=1e-5), loss='binary_crossentropy', metrics=['accuracy'])
print('PHASE 2: Fine-Tuning (No Augmentation, No Callbacks)'); print('='*55)
history_phase2 = model.fit(train_gen, epochs=10, validation_data=val_gen)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history_phase2.history['accuracy'], label='Train', color='#e74c3c'); ax1.plot(history_phase2.history['val_accuracy'], label='Val', color='#3498db')
ax1.set_title('Phase 2 — Accuracy (No Aug, No CB)', fontweight='bold'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(history_phase2.history['loss'], label='Train', color='#e74c3c'); ax2.plot(history_phase2.history['val_loss'], label='Val', color='#3498db')
ax2.set_title('Phase 2 — Loss (No Aug, No CB)', fontweight='bold'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
best_p1 = max(history_phase1.history['val_accuracy']); best_p2 = max(history_phase2.history['val_accuracy'])
last_p2 = history_phase2.history['val_accuracy'][-1]
print(f'Best Phase 2 Val Accuracy: {best_p2*100:.2f}%')
print(f'Last Phase 2 Val Accuracy: {last_p2*100:.2f}%')
print(f'\nNote: Without EarlyStopping, the LAST epoch model is used.')
print(f'The BEST epoch may have been earlier but we cannot go back to it.')

In [ ]:
# 10. Test evaluation
test_loss, test_acc = model.evaluate(test_gen)
print(f'\nTest Accuracy: {test_acc*100:.2f}%'); print(f'Test Loss: {test_loss:.4f}')

In [ ]:
# 11. Confusion Matrix
test_gen.reset()
y_pred = (model.predict(test_gen) > 0.5).astype(int).flatten()
y_true = test_gen.classes; class_names = list(test_gen.class_indices.keys())
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=class_names, yticklabels=class_names, annot_kws={'size': 16})
plt.title('Confusion Matrix — InceptionV3 (No Aug, No CB)', fontsize=16, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.show()

In [ ]:
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# 13. Visualize predictions
test_gen.reset()
images, labels = next(test_gen); predictions = model.predict(images)
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i]); true_label = class_names[int(labels[i])]; pred_label = class_names[int(predictions[i] > 0.5)]
    confidence = predictions[i][0] if predictions[i] > 0.5 else 1 - predictions[i][0]
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.1%})', fontsize=10, color=color, fontweight='bold'); ax.axis('off')
plt.suptitle('InceptionV3 Predictions (No Aug, No CB)', fontsize=16, fontweight='bold'); plt.tight_layout(); plt.show()

In [ ]:
model.save('inceptionv3_leukemia_nocb_final.keras')
print('Model saved.')